# ML Challenge Overfit et Debordés
## Import Packages


In [30]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor 
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import OneHotEncoder

## Data Importation

In [31]:
X_train = pd.read_csv("data/challenge_train_features.csv", index_col=0)
y_train = pd.read_csv("data/challenge_train_revenue.csv", index_col=0)
X_test = pd.read_csv("data/challenge_test_features.csv", index_col=0)

## Preprocessing :

In [32]:
# Preprocessing functions

# Clip popularity scores at 20
def clip_popularity(X):
    X = X.copy()
    X['popularity_score'] = X['popularity_score'].clip(upper=20)
    return X

# Create a binary indicator for zero budget (nan values)
def budget_missing_indicator(X):
    X= X.copy()
    X['budget_is_zero'] = (X['budget'] == 0).astype(int)
    return X

def log_budget(X):
    X = X.copy()
    X['budget'] = np.log1p(X['budget'].clip(lower=0))
    return X

def budget_0_to_nan(X):
    X= X.copy()
    X['budget_nonzero'] = X['budget'].replace(0, np.nan)
    return X

def collection_to_binary(X):
    X = X.copy()
    X['has_collection'] = X['collection'].notna().astype(int)
    return X

def english_to_binary(X):
    """
    Convert language column to binary (0/1) where:
    1 = movie is in English ('en')
    0 = movie is in another language
    """
    X = X.copy()
    X['language'] = (X['language'] == 'en').astype(int)
    return X

def US_to_binary(X):
    """
    Convert country column to binary (0/1) where:
    1 = movie is from US
    0 = movie is from another country
    """
    X = X.copy()
    X['country'] = (X['country'] == 'US').astype(int)
    return X

def get_frequent_genres(X, min_occurrence=150):
    """
    Get list of genres that appear more than min_occurrence times
    Args:
        X: DataFrame containing 'genre' column
        min_occurrence: minimum number of occurrences to keep a genre
    Returns:
        List of frequent genres + "Other"
    """
    X = X.copy()
    X["genres_list"] = X["genre"].apply(lambda x: x.split(",") if isinstance(x, str) else [])
    genre_counts = X["genres_list"].explode().value_counts()
    selected_genres = genre_counts[genre_counts >= min_occurrence].index.tolist()
    return selected_genres + ["Other"]

def encode_genres(X, selected_genres):
    """
    Encode genres into binary columns based on a predefined list of genres
    
    Args:
        X: DataFrame containing 'genre' column
        selected_genres: List of genres to encode (including "Other")
        
    Returns:
        DataFrame with binary columns for each genre
    """
    X = X.copy()
    # Split genres into lists
    X["genres_list"] = X["genre"].apply(lambda x: x.split(",") if isinstance(x, str) else [])
    
    # Map less frequent genres to "Other"
    X["genres_list"] = X["genres_list"].apply(
        lambda lst: [g if g in selected_genres else "Other" for g in lst]
    )
    
    # Create binary columns for each genre
    for genre in selected_genres:
        X[genre] = X["genres_list"].apply(lambda lst: int(genre in lst))
    
    return X

### Transformation to the train and test df

In [33]:
X_train = clip_popularity(X_train)
X_test = clip_popularity(X_test)

X_train = budget_missing_indicator(X_train)
X_test = budget_missing_indicator(X_test)

X_train = budget_0_to_nan(X_train)
X_test = budget_0_to_nan(X_test)

X_train = collection_to_binary(X_train)
X_test = collection_to_binary(X_test)

X_train = english_to_binary(X_train)
X_test = english_to_binary(X_test)

X_train = US_to_binary(X_train)
X_test = US_to_binary(X_test)

selected_genres = get_frequent_genres(X_train, min_occurrence=150)
X_train = encode_genres(X_train, selected_genres)
X_test = encode_genres(X_test, selected_genres)

# X_train = log_budget(X_train)
# X_test = log_budget(X_test)

# Final feature set
feature_columns = ['popularity_score', 'budget', 'budget_is_zero', 'has_collection', 'language', 'country','length'] + selected_genres
X_train_final = X_train[feature_columns]
X_test_final = X_test[feature_columns]

## Training models :

XGB BOOST

In [34]:
model_xgb = XGBRegressor(
    n_estimators=2000,
    learning_rate=0.03,
    max_depth=6,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_lambda=1.0,
    min_child_weight=1.0,
    objective="reg:squarederror",
    tree_method="hist",
    random_state=1,
    n_jobs=-1
)
y_train_log = np.log1p(y_train)
model_xgb.fit(X_train_final, y_train_log)

y_pred_log= model_xgb.predict(X_test_final)
y_pred = np.expm1(y_pred_log).clip(0, None)




saving result in a txt file :

In [35]:
pred_str = ",".join([str(int(p)) for p in y_pred])  

with open("xgboost.txt", "w") as f:
    f.write(pred_str)

CATBOOST

In [ ]:
from catboost import CatBoostRegressor, Pool


train_pool = Pool(X_train_final, y_train_log)

model = CatBoostRegressor(
    iterations=800,
    learning_rate=0.05,
    depth=6,
    loss_function='RMSE', 
    eval_metric='MSLE',
    random_seed=2,
    verbose=100
)

model.fit(train_pool)

# Prédictions
y_pred_log = model.predict(X_test_final)
y_pred = np.expm1(y_pred_log).clip(0, None)  

CatBoostError: tools/enum_parser/enum_serialization_runtime/enum_runtime.cpp:70: Key 'RMSLE' not found in enum ELossFunction. Valid options are: 'Logloss', 'CrossEntropy', 'CtrFactor', 'Focal', 'RMSE', 'LogCosh', 'Lq', 'MAE', 'Quantile', 'MultiQuantile', 'Expectile', 'LogLinQuantile', 'MAPE', 'Poisson', 'MSLE', 'MedianAbsoluteError', 'SMAPE', 'Huber', 'Tweedie', 'Cox', 'RMSEWithUncertainty', 'MultiClass', 'MultiClassOneVsAll', 'PairLogit', 'PairLogitPairwise', 'YetiRank', 'YetiRankPairwise', 'QueryRMSE', 'GroupQuantile', 'QuerySoftMax', 'QueryCrossEntropy', 'StochasticFilter', 'LambdaMart', 'StochasticRank', 'PythonUserDefinedPerObject', 'PythonUserDefinedMultiTarget', 'UserPerObjMetric', 'UserQuerywiseMetric', 'R2', 'NumErrors', 'FairLoss', 'AUC', 'Accuracy', 'BalancedAccuracy', 'BalancedErrorRate', 'BrierScore', 'Precision', 'Recall', 'F1', 'TotalF1', 'F', 'MCC', 'ZeroOneLoss', 'HammingLoss', 'HingeLoss', 'Kappa', 'WKappa', 'LogLikelihoodOfPrediction', 'NormalizedGini', 'PRAUC', 'Pai

saving results in a txt file

In [ ]:
pred_str = ",".join([str(int(p)) for p in y_pred])


with open("catboost.txt", "w") as f:
    f.write(pred_str)

y_pred = 0

            revenue
count  2.000000e+03
mean   6.683178e+07
std    1.384006e+08
min    1.000000e+00
25%    2.586293e+06
50%    1.720186e+07
75%    6.985050e+07
max    1.519558e+09
